In [1]:
import gradio as gr
import joblib

# Attempt to load the trained model
try:
    # This must match the filename saved by train.py
    model = joblib.load('best_phishing_detector.pkl')
    print("Model loaded successfully!")
except FileNotFoundError:
    print("Warning: 'best_phishing_detector.pkl' not found.")
    print("Please make sure you have successfully run 'train.py' first to generate the model.")
    model = None

def predict_url(url):
    """
    Takes a URL string, feeds it into the model, and returns a formatted result.
    """
    if not url.strip():
        return "Please enter a valid URL."
        
    if model is None:
        return "Error: Model not found. You must train the model first by running train.py."
        
    try:
        raw_pred = model.predict([url])[0]
        
        # It's Linear Regression (outputs continuous decimals instead of probabilities)
        prediction = 1 if raw_pred >= 0.5 else 0
        
        # Cap the "confidence" between 0 and 100
        mock_prob = max(0, min(1, raw_pred))
        confidence = mock_prob * 100 if prediction == 1 else (1 - mock_prob) * 100
            
        # Format the output based on the prediction
        if prediction == 1:
            return f"🚨 FRAUDULENT (Phishing) 🚨\nThe model is {confidence:.1f}% confident this is a bad URL."
        else:
            return f"✅ GENUINE (Safe) ✅\nThe model is {confidence:.1f}% confident this is a safe URL."
            
    except Exception as e:
        return f"An error occurred during prediction: {str(e)}"

# Create the Gradio interface
demo = gr.Interface(
    fn=predict_url,
    inputs=gr.Textbox(
        lines=2, 
        placeholder="e.g., http://paypal-secure-login-update.com", 
        label="Enter URL to analyze"
    ),
    outputs=gr.Textbox(label="Prediction Result"),
    title="🎣 Phishing URL Detector",
    description="Enter a URL to check if it's a safe website or a malicious phishing attempt. The model uses a Machine Learning Linear Regression pipeline trained on real-world phishing data.",
    examples=[
        ["https://www.google.com"],
        ["http://secure-banking-login-verify-account.com/update"],
        ["https://github.com/torvalds/linux"]
    ],
    theme=gr.themes.Soft()
)

if __name__ == "__main__":
    # Launch the web app
    print("Starting the web interface...")
    demo.launch(server_name="0.0.0.0", server_port=7860)

Model loaded successfully!
Starting the web interface...
Running on local URL:  http://0.0.0.0:7860

To create a public link, set `share=True` in `launch()`.
